# 05 – End-to-End Inference on Videos

This notebook runs the full two-stage pipeline on a video:

1. **Detection** – YOLOv8 locates every trash item in the frame.
2. **Classification** – EfficientNetV2-S identifies the material type of each
   detected crop.
3. **Overlay** – material label + confidence score is drawn on the frame.
4. **Export** – annotated frames are written to an output video file.

## 1 · Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("__file__").resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import cv2
import numpy as np
import torch
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

from trash_detection.viz import draw_bbox

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 2 · Load Models

In [ ]:
from ultralytics import YOLO
from torchvision.models import efficientnet_v2_s
from torch import nn

# ── Paths – update these to your actual model files ───────────────────────────
DETECTOR_PATH    = REPO_ROOT / "runs" / "detect" / "trash_detector_v1" / "weights" / "best.pt"
CLASSIFIER_PATH  = REPO_ROOT / "models" / "classification" / "classifier_checkpoint_full.pt"
INPUT_VIDEO_PATH = REPO_ROOT / "data" / "videos" / "sample.mp4"  # ← set your video path
OUTPUT_VIDEO_DIR = REPO_ROOT / "outputs"
OUTPUT_VIDEO_DIR.mkdir(parents=True, exist_ok=True)

# ── Load YOLO detector ────────────────────────────────────────────────────────
if DETECTOR_PATH.exists():
    detector = YOLO(str(DETECTOR_PATH))
    print(f"Detector loaded: {DETECTOR_PATH.name}")
else:
    raise FileNotFoundError(f"Detector weights not found: {DETECTOR_PATH}")

# ── Load EfficientNetV2-S classifier ─────────────────────────────────────────
if CLASSIFIER_PATH.exists():
    ckpt = torch.load(CLASSIFIER_PATH, map_location=DEVICE)
    CLASS_NAMES = ckpt["class_names"]
    NUM_CLASSES = ckpt["num_classes"]

    classifier = efficientnet_v2_s(weights=None)
    in_features = classifier.classifier[1].in_features
    classifier.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, NUM_CLASSES),
    )
    classifier.load_state_dict(ckpt["model_state_dict"])
    classifier = classifier.to(DEVICE).eval()
    print(f"Classifier loaded. Classes: {CLASS_NAMES}")
else:
    raise FileNotFoundError(f"Classifier checkpoint not found: {CLASSIFIER_PATH}")

## 3 · Per-Frame Pipeline

```
frame → YOLO detect → [bbox₁, bbox₂, …]
                             │
                    crop + pad + resize 224×224
                             │
                   EfficientNetV2-S classify
                             │
                    overlay label + confidence
```

In [ ]:
# Preprocessing transform for the classifier
clf_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Colour palette for material classes (BGR)
CLASS_COLORS = {
    "plastic" : (0, 165, 255),   # orange
    "glass"   : (255, 255,   0), # cyan
    "metal"   : (128, 128, 128), # grey
    "paper"   : (0, 255, 255),   # yellow
    "other"   : (0,   0, 255),   # red
}


def classify_crop(
    frame: np.ndarray,
    x1: int, y1: int, x2: int, y2: int,
    padding: float = 0.10,
) -> tuple[str, float]:
    """Crop, pad, and classify a single bounding box."""
    ih, iw = frame.shape[:2]
    bw, bh = x2 - x1, y2 - y1
    px, py = int(bw * padding), int(bh * padding)
    cx1, cy1 = max(0, x1 - px), max(0, y1 - py)
    cx2, cy2 = min(iw, x2 + px), min(ih, y2 + py)

    crop_bgr = frame[cy1:cy2, cx1:cx2]
    if crop_bgr.size == 0:
        return "other", 0.0

    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    tensor = clf_transform(Image.fromarray(crop_rgb)).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probs = torch.softmax(classifier(tensor), dim=1)[0].cpu().numpy()

    pred_idx = int(np.argmax(probs))
    return CLASS_NAMES[pred_idx], float(probs[pred_idx])


def process_frame(
    frame: np.ndarray,
    conf_threshold: float = 0.25,
) -> np.ndarray:
    """Run detection + classification on a single frame, return annotated copy."""
    annotated = frame.copy()
    results = detector.predict(frame, conf=conf_threshold, imgsz=640, verbose=False)

    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        det_conf = float(box.conf[0])

        material, clf_conf = classify_crop(frame, x1, y1, x2, y2)
        color = CLASS_COLORS.get(material, (0, 255, 0))
        label = f"{material} {clf_conf:.0%} (det {det_conf:.0%})"
        draw_bbox(annotated, x1, y1, x2, y2, label=label, color=color)

    return annotated


print("Pipeline helper functions defined.")

## 4 · Process Video

In [ ]:
import time

if not INPUT_VIDEO_PATH.exists():
    print(f"Input video not found: {INPUT_VIDEO_PATH}")
    print("Update INPUT_VIDEO_PATH in Section 2.")
else:
    cap = cv2.VideoCapture(str(INPUT_VIDEO_PATH))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out_path = OUTPUT_VIDEO_DIR / f"annotated_{INPUT_VIDEO_PATH.stem}.mp4"
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, fps, (width, height))

    print(f"Processing {total} frames ({width}x{height} @ {fps:.1f} fps) …")
    t0 = time.time()
    frame_count = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        annotated = process_frame(frame, conf_threshold=0.25)
        writer.write(annotated)
        frame_count += 1
        if frame_count % 50 == 0:
            elapsed = time.time() - t0
            print(f"  {frame_count}/{total} frames | {elapsed:.1f}s elapsed")

    cap.release()
    writer.release()
    elapsed = time.time() - t0
    print(f"\nDone – {frame_count} frames in {elapsed:.1f}s ({frame_count/elapsed:.1f} fps)")
    print(f"Output saved → {out_path}")

## 5 · Display Results

Sample a few annotated frames and display them as a grid.

In [ ]:
from trash_detection.viz import show_image_grid

SAMPLE_FRAMES_PATH = INPUT_VIDEO_PATH  # re-open the original video
SAMPLE_INDICES = [0, 30, 60, 90, 120, 150, 180, 210]  # frames to visualise

if INPUT_VIDEO_PATH.exists():
    cap = cv2.VideoCapture(str(SAMPLE_FRAMES_PATH))
    sample_frames = []
    sample_titles = []

    for idx in SAMPLE_INDICES:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if ok:
            annotated = process_frame(frame, conf_threshold=0.25)
            sample_frames.append(annotated)
            sample_titles.append(f"frame {idx}")

    cap.release()

    if sample_frames:
        show_image_grid(sample_frames, titles=sample_titles, cols=4)
    else:
        print("Could not read sample frames from the video.")
else:
    print("Input video not available – skipping display.")